In [1]:
import gc
import numpy as np
import pandas as pd
from dataclasses import dataclass, replace
from typing import Optional, Tuple, Dict


# ============================================================
# 0) Small utilities
# ============================================================

def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))


def _fmt_df(df: pd.DataFrame, nan_dash_cols=()) -> str:
    """
    Console-friendly table printing with 3-decimal rounding.
    For columns in nan_dash_cols, NaN is printed as '–'.
    """
    df_disp = df.copy()
    formatters = {}

    for c in df_disp.columns:
        if pd.api.types.is_numeric_dtype(df_disp[c]):
            if c in nan_dash_cols:
                formatters[c] = lambda v: "–" if pd.isna(v) else f"{v:.3f}"
            else:
                formatters[c] = lambda v: f"{v:.3f}"
    return df_disp.to_string(index=False, formatters=formatters)


# ============================================================
# 1) Stylized simulation parameters (paper notation)
# ============================================================

@dataclass(frozen=True)
class StylizedParams:
    # population / horizon
    N: int = 100               # number of clients
    T: int = 300               # rounds
    theta: float = 0.30        # gaming fraction (paper: 30%)

    # reward / participation
    r: float = 1.0             # reward scale (multiplies metric)
    b: float = 0.2             # reward bias
    mu_cost: float = 0.4       # mean participation cost
    sigma_cost: float = 0.1    # std participation cost
    beta: float = 4.0          # logistic slope
    tau: float = 0.0           # participation bias/threshold

    # auditing / sanctions
    B: int = 10                # audit budget (max audited per round)
    alpha_penalty: float = 0.7 # sanction strength α_penalty
    p_detect: float = 0.7      # detection prob when audited & gaming

    # metric design (public/private mix)
    rho_pub: float = 0.6       # public-metric weight ρ_pub
    sigma_pub: float = 0.01    # public noise
    sigma_priv: float = 0.01   # private noise
    delta_metric: float = 0.3  # metric inflation term (gaming share)

    # welfare impact of gaming
    gamma: float = 0.7         # welfare harm coefficient γ

    # reproducibility
    seed: Optional[int] = 42   # RNG seed


# ============================================================
# 2) Initialization
# ============================================================

def init_population(p: StylizedParams) -> Dict[str, np.ndarray]:
    rng = np.random.default_rng(p.seed)

    N = p.N
    n_gaming = int(np.round(p.theta * N))

    is_gaming = np.zeros(N, dtype=bool)
    if n_gaming > 0:
        idx = rng.choice(N, size=n_gaming, replace=False)
        is_gaming[idx] = True

    cost = rng.normal(loc=p.mu_cost, scale=p.sigma_cost, size=N)
    cost = np.clip(cost, 0.05, None)

    return {"is_gaming": is_gaming, "cost": cost, "rng": rng}


# ============================================================
# 3) One round dynamics (kept RNG call order to match paper tables)
# ============================================================

def step_round(p: StylizedParams, state: Dict[str, np.ndarray], M_prev: float) -> Tuple[float, float, float]:
    is_gaming = state["is_gaming"]
    cost = state["cost"]
    rng = state["rng"]

    N = p.N

    # --- participation decision (expected penalty approximation) ---
    q_audit = p.B / max(N, 1)  # approx audit probability
    expected_penalty = p.alpha_penalty * q_audit * p.p_detect

    profit = p.b + p.r * M_prev - cost - expected_penalty * is_gaming.astype(float)
    p_join = sigmoid(p.beta * (profit - p.tau))

    participate = rng.random(size=N) < p_join
    participants_idx = np.flatnonzero(participate)
    n_participants = int(participants_idx.size)

    if n_participants > 0:
        G = int(np.sum(is_gaming[participants_idx]))
        H = int(n_participants - G)
    else:
        G = 0
        H = 0

    x = n_participants / max(N, 1)

    # --- welfare ---
    W = (H - p.gamma * G) / max(N, 1)
    W = float(np.clip(W, 0.0, 1.0))

    # --- metric (public/private mix) ---
    noise_pub = rng.normal(0.0, p.sigma_pub)
    noise_priv = rng.normal(0.0, p.sigma_priv)

    M_pub = W + p.delta_metric * (G / max(N, 1)) + noise_pub
    M_priv = W + noise_priv

    M = p.rho_pub * M_pub + (1.0 - p.rho_pub) * M_priv
    M = float(np.clip(M, 0.0, 1.0))

    # --- auditing & detection (kept for RNG-consumption consistency) ---
    if n_participants > 0 and p.B > 0:
        k = min(p.B, n_participants)
        audited_idx = rng.choice(participants_idx, size=k, replace=False)
        is_gaming_audited = is_gaming[audited_idx]
        _ = rng.random(size=k) < (p.p_detect * is_gaming_audited.astype(float))
        # We do not store penalized/audit logs because tables 1–3 do not use them.

    return W, x, M


# ============================================================
# 4) Steady-state averages (post burn-in) without storing per-round logs
# ============================================================

def steady_state_means(p: StylizedParams, burn_in: int = 100, M0: float = 0.6) -> Tuple[float, float, float]:
    state = init_population(p)
    M_prev = M0

    W_sum = 0.0
    x_sum = 0.0
    M_sum = 0.0
    cnt = 0

    for t in range(p.T):
        W, x, M = step_round(p, state, M_prev)
        M_prev = M

        if t >= burn_in:
            W_sum += W
            x_sum += x
            M_sum += M
            cnt += 1

    # cleanup
    del state
    gc.collect()

    return W_sum / cnt, x_sum / cnt, M_sum / cnt


def price_of_gaming(p_aligned: StylizedParams, p_gaming: StylizedParams, burn_in: int = 100) -> Tuple[float, float, float, float, float, float, float]:
    W_a, x_a, M_a = steady_state_means(p_aligned, burn_in=burn_in)
    W_g, x_g, M_g = steady_state_means(p_gaming, burn_in=burn_in)
    PoG = np.nan if W_a <= 0 else (W_a - W_g) / W_a
    return W_a, x_a, M_a, W_g, x_g, M_g, PoG


# ============================================================
# 5) Paper tables 1–3
# ============================================================

def make_table_1(base: StylizedParams, burn_in: int = 100) -> pd.DataFrame:
    p_aligned = replace(base, theta=0.0)
    p_gaming = base

    W_a, x_a, M_a, W_g, x_g, M_g, PoG = price_of_gaming(p_aligned, p_gaming, burn_in=burn_in)

    df = pd.DataFrame([
        {"Scenario": "Aligned", "W": W_a, "x": x_a, "M": M_a, "M − W": (M_a - W_a), "PoG": np.nan},
        {"Scenario": "Gaming",  "W": W_g, "x": x_g, "M": M_g, "M − W": (M_g - W_g), "PoG": PoG},
    ])
    return df


def make_table_2(base: StylizedParams, alphas, burn_in: int = 100) -> pd.DataFrame:
    # Aligned is independent of alpha (no gaming), so compute once for efficiency.
    p_aligned = replace(base, theta=0.0)
    W_a, x_a, M_a = steady_state_means(p_aligned, burn_in=burn_in)

    rows = []
    for a in alphas:
        p_game = replace(base, alpha_penalty=float(a))
        W_g, x_g, M_g = steady_state_means(p_game, burn_in=burn_in)
        PoG = (W_a - W_g) / W_a

        rows.append({
            "α_penalty": float(a),
            "W_game": W_g,
            "x_game": x_g,
            "PoG": PoG,
        })

        del p_game
        gc.collect()

    return pd.DataFrame(rows)


def make_table_3(base: StylizedParams, rhos, burn_in: int = 100) -> pd.DataFrame:
    rows = []
    for rho in rhos:
        rho = float(rho)
        p_aligned = replace(base, theta=0.0, rho_pub=rho)
        p_game = replace(base, rho_pub=rho)

        W_a, x_a, M_a = steady_state_means(p_aligned, burn_in=burn_in)
        W_g, x_g, M_g = steady_state_means(p_game, burn_in=burn_in)
        PoG = (W_a - W_g) / W_a

        rows.append({
            "ρ_pub": rho,
            "W_game": W_g,
            "x_game": x_g,
            "PoG": PoG,
            "M_game": M_g,
            "M_game − W_game": (M_g - W_g),
        })

        del p_aligned, p_game
        gc.collect()

    return pd.DataFrame(rows)


# ============================================================
# 6) Script entry: print ONLY paper Tables 1–3
# ============================================================

if __name__ == "__main__":
    # Baseline parameters used for Tables 1–3 (paper stylized simulator)
    base = StylizedParams()

    # Table 1
    df1 = make_table_1(base, burn_in=100)
    print("Table 1: Baseline aligned versus gaming scenarios (steady-state averages over post–burn-in rounds).")
    print(_fmt_df(df1, nan_dash_cols=("PoG",)))
    print()

    # Table 2
    alphas = [0.3, 0.5, 0.7, 1.0, 1.5]
    df2 = make_table_2(base, alphas=alphas, burn_in=100)
    print("Table 2: Effect of sanction strength α_penalty on gaming scenarios (steady-state averages).")
    print(_fmt_df(df2))
    print()

    # Table 3
    rhos = [1.0, 0.8, 0.6, 0.4, 0.2]
    df3 = make_table_3(base, rhos=rhos, burn_in=100)
    print("Table 3: Effect of public-metric weight ρ_pub on gaming scenarios (steady-state averages).")
    print(_fmt_df(df3))

Table 1: Baseline aligned versus gaming scenarios (steady-state averages over post–burn-in rounds).
Scenario     W     x     M  M − W   PoG
 Aligned 0.952 0.952 0.952 -0.000   NaN
  Gaming 0.325 0.638 0.358  0.033 0.658

Table 2: Effect of sanction strength α_penalty on gaming scenarios (steady-state averages).
α_penalty W_game x_game   PoG
    0.300  0.316  0.637 0.668
    0.500  0.319  0.636 0.665
    0.700  0.325  0.638 0.658
    1.000  0.332  0.636 0.651
    1.500  0.340  0.632 0.643

Table 3: Effect of public-metric weight ρ_pub on gaming scenarios (steady-state averages).
ρ_pub W_game x_game   PoG M_game M_game − W_game
1.000  0.341  0.673 0.642  0.399           0.058
0.800  0.331  0.652 0.652  0.376           0.045
0.600  0.325  0.638 0.658  0.358           0.033
0.400  0.317  0.621 0.667  0.339           0.021
0.200  0.309  0.603 0.675  0.320           0.011
